# 01 · Ingestão / Download dos Microdados

**Objetivo:** obter os microdados RAIS (vínculos) e Novo CAGED para os anos do escopo.

**Entradas:** `config.yaml` (anos, URLs, modo sintético).  
**Saídas:** arquivos em `data/raw/{RAIS,CAGED}/{ano}/` (parquet).

**Premissas:** acesso público estável.  
**Decisões:** download idempotente com cache; *fallback* para amostra sintética.  
**Limitações:** volume grande (dezenas de GB em 5 anos × 27 UF) — use `ufs_subset`
ou `synthetic_mode` para protótipo.

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import load_config, anos_validos
cfg = load_config()
print('Raiz:', cfg['root'], '| modo sintético:', cfg['synthetic_mode'])

In [ ]:
from src import io_utils
raw = cfg['abs']['raw']
anos = cfg['anos']

if cfg['synthetic_mode']:
    # Modo sintético: gera amostras com o schema bruto e salva como parquet.
    s = cfg['synthetic']
    for ano in anos:
        rais = io_utils.gen_rais_ano(ano, s['n_vinculos_por_ano'], s['seed'])
        caged = io_utils.gen_caged_ano(ano, s['n_movs_caged_por_ano'], s['seed'])
        (raw / 'RAIS' / str(ano)).mkdir(parents=True, exist_ok=True)
        (raw / 'CAGED' / str(ano)).mkdir(parents=True, exist_ok=True)
        rais.to_parquet(raw / 'RAIS' / str(ano) / 'rais.parquet', index=False)
        caged.to_parquet(raw / 'CAGED' / str(ano) / 'caged.parquet', index=False)
        print(f'[sintético] {ano}: RAIS={len(rais):,} CAGED={len(caged):,}')
else:
    # Modo REAL: baixa os vínculos RAIS por região do FTP do PDET/MTE.
    # Arquivos: {rais_base}/{ano}/RAIS_VINC_PUB_<REGIAO>.7z  (CSV ',' latin-1).
    base = cfg['urls']['rais_base']
    regioes = cfg['rais']['regioes']
    for ano in anos:
        for reg in regioes:
            nome = f'RAIS_VINC_PUB_{reg}.7z'
            url = f'{base}/{ano}/{nome}'
            dest = raw / 'RAIS' / str(ano) / nome
            print(f'[real] baixando {ano}/{reg} ...', flush=True)
            io_utils.download_ftp(url, dest)            # idempotente (cache)
            print(f'        ok {dest.name} ({dest.stat().st_size/1e6:.0f} MB)', flush=True)
    print('Download RAIS concluído. (A extração ocorre no nb02, sob demanda.)')
    # CAGED real (componente recente) — opcional; o cálculo de taxas usa a RAIS.
    # TODO: decodificar o layout do Novo CAGED (CAGEDMOV{AAAAMM}.7z) se for usar fluxos.

In [ ]:
# Conferência rápida do que foi gravado
if cfg['synthetic_mode']:
    for fonte in ['RAIS', 'CAGED']:
        files = sorted((raw / fonte).rglob('*.parquet'))
        print(fonte, '->', len(files), 'arquivo(s)')
        if files: display(pd.read_parquet(files[0]).head(3))
else:
    z = sorted((raw / 'RAIS').rglob('*.7z'))
    print(len(z), 'arquivos .7z baixados:')
    for c in z:
        print(' ', c.relative_to(raw), f'-> {c.stat().st_size/1e6:.0f} MB')